# Kaggle: train Model A1 on GPU

Trains A1 across all four C-MAPSS subsets and three seeds, then saves the
checkpoints and the run registry as a Kaggle output dataset.

**Before running:** switch the accelerator to a GPU (Settings -> Accelerator ->
GPU T4 x2 or P100) and turn Internet **on** (needed to clone the repo and fetch
C-MAPSS).

The repository is private, so the clone needs a GitHub token. Add it as a Kaggle
secret named `GITHUB_TOKEN` (Add-ons -> Secrets) rather than pasting it into a
cell -- notebook source is saved with the notebook.


In [ ]:
import subprocess, sys, os, time

print("=== GPU ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "no GPU detected -- switch the accelerator on in Settings")

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## Clone the repository

In [ ]:
from kaggle_secrets import UserSecretsClient

REPO = "harvik07/twinmind"
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if not os.path.exists("/kaggle/working/twinmind"):
    subprocess.run(
        ["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git",
         "/kaggle/working/twinmind"],
        check=True,
    )
os.chdir("/kaggle/working/twinmind")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
del token  # do not leave it in the kernel namespace

## Install

`requirements.txt` is a flat file precisely so this cell is one line. Kaggle
already ships torch, numpy, pandas and scikit-learn, so `-e .` mostly just puts
`twinmind` on the path.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import twinmind
print("twinmind", twinmind.__version__, "from", twinmind.__file__)

## Fetch C-MAPSS\n\nAbout 45 MB, then cached by the manifest on a re-run.

In [ ]:
t0 = time.time()
subprocess.run([sys.executable, "scripts/download_data.py", "--dataset", "cmapss"], check=True)
print(f"download took {time.time() - t0:.0f}s")

from twinmind.data.loaders import load_cmapss
for s in ["FD001", "FD002", "FD003", "FD004"]:
    tr = load_cmapss(s, "train")
    print(f"{s}: {len(tr):,} rows, {tr.unit.nunique()} units")

## Estimated runtime

Measured on the development CPU (AMD, 10 threads) at roughly 19 s/epoch for
FD001 and about 3.5x that for FD002/FD004, with early stopping typically at
40-70 epochs. A T4 should be several times faster; the cell below times a single
epoch on this machine so the estimate is measured rather than guessed.


In [ ]:
from twinmind.data.cmapss_prep import prepare_cmapss
from twinmind.models.a1_train import TrainConfig, train_a1
from twinmind.utils.config import load_config

cfg = load_config("data")
device = "cuda" if torch.cuda.is_available() else "cpu"

probe = prepare_cmapss("FD001", cfg.cmapss, seed=42)
t0 = time.time()
_, _ = train_a1(probe, seed=42, lambda_fail=0.5, device=device,
                train_config=TrainConfig(epochs=1, lambda_candidates=(0.5,), mc_passes=2),
                quiet=True)
per_epoch = time.time() - t0
print(f"one FD001 epoch on {device}: {per_epoch:.1f}s")

# FD002/FD004 hold roughly 3.5x the windows of FD001/FD003.
scale = {"FD001": 1.0, "FD002": 3.5, "FD003": 1.2, "FD004": 3.5}
total = sum(per_epoch * s * 55 * 3 for s in scale.values())   # ~55 epochs, 3 seeds
print(f"estimated total for 4 subsets x 3 seeds: {total/60:.0f} min ({total/3600:.1f} h)")

## Train

Three independent seeds per subset. `--tune-lambda` searches {0.1, 0.5, 1.0} on
validation RMSE and triples the cost, so it is off here; run it once for FD001
and reuse the chosen value, which is what the CPU runs did.


In [ ]:
t0 = time.time()
subprocess.run(
    [sys.executable, "scripts/train_health.py",
     "--subset", "all", "--seeds", "3", "--device", device, "--epochs", "100"],
    check=True,
)
print(f"training took {(time.time() - t0)/60:.1f} min on {device}")

## Also train A2 (cheap, CPU-bound)

In [ ]:
subprocess.run([sys.executable, "scripts/download_data.py", "--dataset", "ai4i"], check=True)
subprocess.run([sys.executable, "scripts/train_ai4i.py", "--model", "both", "--seeds", "3"], check=True)

## Results

In [ ]:
subprocess.run([sys.executable, "-m", "twinmind.eval.registry"], check=True)
print(open("reports/results/summary.md", encoding="utf-8").read())

## Save the outputs

Everything a later session needs -- checkpoints, the append-only run log,
per-epoch history and the figures -- copied to `/kaggle/working/outputs`, which
Kaggle saves as an output dataset when the notebook finishes.


In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/outputs")
out.mkdir(exist_ok=True)

for src, name in [("models/a1", "checkpoints"), ("reports/results", "results"),
                  ("reports/figures", "figures")]:
    source = Path(src)
    if source.exists():
        shutil.copytree(source, out / name, dirs_exist_ok=True)
        print(f"{name}: {len(list((out / name).rglob('*')))} files")

subprocess.run([sys.executable, "scripts/make_figures.py"], check=False)
shutil.copytree("reports/figures", out / "figures", dirs_exist_ok=True)
print("\nsaved to", out)

## Bringing the results home

Download the output dataset, then on the development machine:

```bash
cp <downloaded>/results/runs.jsonl reports/results/runs.jsonl
cp -r <downloaded>/checkpoints/* models/a1/
python -m twinmind.eval.registry
```

`runs.jsonl` is append-only, so GPU runs sit alongside the CPU ones with their
own `device` field and both appear in the summary table.
